# Media transformations

This source-only notebook generates disposable raster and video fixtures, then demonstrates `extract_frames`, `crop_image`, and `annotate_image` as separate nonblocking browser calls. Run each inspection in a later cell after its receipt completes. It needs this JupyterLab frontend and a live Python kernel, not a provider or personal media. Keep the kernel working directory under the same disposable Jupyter server root.

Available tools: &`extract_frames` &`crop_image` &`annotate_image`


## Disposable exact PNG and video references


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from hashlib import sha256
from base64 import b64decode
from PIL import Image, ImageDraw

work = Path(mkdtemp(prefix='nbinlineai-transform-', dir=Path.cwd()))
image_path = work / 'source.png'
video_path = work / 'red-blue.webm'
image = Image.new('RGB', (64, 64), 'white')
ImageDraw.Draw(image).rectangle((4, 4, 20, 20), fill='red')
image.save(image_path, format='PNG')
video_path.write_bytes(b64decode('GkXfo59ChoEBQveBAULygQRC84EIQoKEd2VibUKHgQJChYECGFOAZwEAAAAAAAILEU2bdLpNu4tTq4QVSalmU6yBoU27i1OrhBZUrmtTrIHGTbuMU6uEElTDZ1OsggETTbuMU6uEHFO7a1OsggH17AEAAAAAAABZAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAVSalmoCrXsYMPQkBNgIRMYXZmV0GETGF2ZkSJiECfQAAAAAAAFlSua8iuAQAAAAAAAD/XgQFzxYgAAAAAAAAAAZyBACK1nIN1bmSIgQCGhVZfVlA5g4EBI+ODhB3NZQDgkLCBELqBEJqBAlWwhFW5gQESVMNn1HNz0WPAi2PFiAAAAAAAAAABZ8icRaOHRU5DT0RFUkSHj0xhdmMgbGlidnB4LXZwOWfIoUWjiERVUkFUSU9ORIeTMDA6MDA6MDIuMDAwMDAwMDAwAB9DtnVAg+eBAKOugQAAgIJJg0IAAPAA9gA4JBwYAAAAIAAAEL///8jXgDH////s/IcP////bIwAAKOSgQH0AIYAQJKcAEAAAAIAAEJAo6aBA+gAhgBAkpwAQAAAAgAAAMn///5zlAGP////2yMBw////jFQAKOSgQXcAIYAQJKcAEAAAAIAAEJAHFO7a5G7j7OBALeK94EB8YIBbPCBAw=='))
image_ref = {'path': image_path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(image_path.read_bytes()).hexdigest()}
video_ref = {'path': video_path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(video_path.read_bytes()).hexdigest()}
print('Disposable exact PNG and 16×16 two-color VP9 clip:', image_ref, video_ref)


## extract_frames

Request two exact positions in the generated two-second red/blue clip. The browser must actually decode and seek; the receipt later contains PIL images plus descriptors with requested and actual times. Memory-only extraction writes no sidecar.


In [ ]:
from nbinlineai.tools import extract_frames
frames = extract_frames(video_ref, [0.25, 1.25])
frames


In [ ]:
print(frames.status, frames.error)
if frames.status == 'completed':
    assert len(frames.result) == len(frames.media) == 2
    colors = [item.getpixel((8, 8)) for item in frames.result]
    assert all(item.size == (16, 16) for item in frames.result)
    assert colors[0][0] > colors[0][2] + 50 and colors[1][2] > colors[1][0] + 50
    assert [item['source_sha256'] for item in frames.media] == [video_ref['sha256']] * 2
    assert 0 <= frames.media[0]['actual_seconds'] <= 0.5
    assert 1 <= frames.media[1]['actual_seconds'] <= 1.5
    print('Verified decoded red and blue frames:', colors, [item['actual_seconds'] for item in frames.media])


## crop_image

Crop the generated PNG to a new 16×16 image and explicitly save it. The source remains 64×64; the saved derivative has its own JSON provenance sidecar.


In [ ]:
from nbinlineai.tools import crop_image
crop = crop_image(image_ref, 4, 4, 16, 16, save_to=f'{work.name}/crop.png')
crop


In [ ]:
print(crop.status, crop.error)
if crop.status == 'completed':
    import json
    assert crop.result.size == (16, 16)
    assert crop.result.getpixel((2, 2))[:3] == (255, 0, 0)
    assert image_path.exists() and Image.open(image_path).size == (64, 64)
    sidecar = json.loads(Path(crop.media['sidecar_path']).read_text())
    assert sidecar['source_sha256'] == image_ref['sha256']
    assert sidecar['transform'] == 'crop_image'
    print(crop.result.size, crop.media['path'], crop.media['sidecar_path'])


## annotate_image

Create a distinct PNG with an opaque black redaction over part of the original red square, plus a rectangle, arrow, and short text. Inspect a redacted pixel and the unchanged source pixel later. This memory-only derivative has no sidecar.


In [ ]:
from nbinlineai.tools import annotate_image
shapes = [
    {'type': 'redaction', 'x': 5, 'y': 5, 'width': 8, 'height': 8},
    {'type': 'rectangle', 'x': 22, 'y': 5, 'width': 16, 'height': 12, 'color': '#1976d2'},
    {'type': 'arrow', 'x1': 3, 'y1': 40, 'x2': 25, 'y2': 40, 'color': '#e53935'},
    {'type': 'text', 'x': 30, 'y': 35, 'text': 'demo', 'font_size': 12, 'color': '#000000'},
]
annotated = annotate_image(image_ref, shapes)
annotated


In [ ]:
print(annotated.status, annotated.error)
if annotated.status == 'completed':
    assert annotated.result.size == (64, 64)
    assert annotated.result.getpixel((7, 7))[:3] == (0, 0, 0)
    assert Image.open(image_path).getpixel((7, 7)) == (255, 0, 0)
    assert annotated.media['source_sha256'] == image_ref['sha256']
    assert annotated.media['transform'] == 'annotate_image'
    print('Opaque redaction changed derivative pixels; source stayed red')


## Clean up disposable files

After inspecting the receipts, remove only this notebook’s generated files and empty folder. Python PIL images already delivered remain in memory until you discard them.


In [ ]:
for path in (work / 'crop.png.json', work / 'crop.png', image_path, video_path):
    path.unlink(missing_ok=True)
work.rmdir()
print('Disposable transform files removed')
